# YOLOv8 Arrow Detection Training

This notebook trains a YOLOv8 model on the `arrow_dataset` dataset and runs validation + sample predictions.

In [ ]:
# Optional (run if dependencies are not installed yet)
# !pip install -r requirements.txt

In [1]:
from pathlib import Path
import yaml
from ultralytics import YOLO

In [2]:
# Dataset configuration
DATA_YAML = Path('arrow_dataset/data.yaml')
assert DATA_YAML.exists(), f'Missing dataset YAML: {DATA_YAML.resolve()}'

with open(DATA_YAML, 'r') as f:
    data_cfg = yaml.safe_load(f)

print('Dataset classes:', data_cfg['names'])
print('Number of classes:', data_cfg['nc'])
print('Resolved data.yaml path:', DATA_YAML.resolve())

Dataset classes: ['down', 'left', 'right', 'up']
Number of classes: 4
Resolved data.yaml path: /home/xd/Documents/py_codes/arrows/arrow_dataset/data.yaml


In [3]:
# Training settings (edit as needed)
MODEL_WEIGHTS = 'yolov8n.pt'  # small/fast baseline
EPOCHS = 50
IMGSZ = 640
BATCH = 16
DEVICE = 0  # set to 'cpu' if no GPU
PROJECT = 'runs'
RUN_NAME = 'arrow_yolov8n'

In [4]:
# Train YOLOv8
model = YOLO(MODEL_WEIGHTS)
train_results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    project=PROJECT,
    name=RUN_NAME
)
train_results

Ultralytics 8.4.19 🚀 Python-3.12.3 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3060, 11887MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=arrow_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=arrow_yolov8n, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100

KeyboardInterrupt: 

In [11]:
# Validate using best checkpoint
# best_model_path = Path(PROJECT) / RUN_NAME / 'weights' / 'last.pt'
best = "/home/xd/Documents/open_src/ComfyUI/runs/detect/runs/arrow_yolov8n/weights/best.pt"
best_model = YOLO(best)
val_metrics = best_model.val(data=str(DATA_YAML))
val_metrics

Ultralytics 8.4.19 🚀 Python-3.12.3 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3060, 11887MiB)
Model summary (fused): 73 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3086.3±1332.3 MB/s, size: 38.6 KB)
val: Scanning /home/xd/Documents/py_codes/arrows/arrow_dataset/valid/labels.cache... 2001 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2001/2001 524.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 126/126 12.4it/s 10.1s0.1s
                   all       2001       7998      0.995      0.997      0.995      0.925
                  down       1337       1966      0.998      0.998      0.995      0.915
                  left       1377       1999      0.993      0.998      0.995      0.931
                 right       1375       1995      0.998      0.991      0.995      0.929
                    up       1319       2038      0.989      0.999      0.994  

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7e775bcaf590>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0

In [12]:
# Run predictions on test images
pred_results = best_model.predict(
    source='arrow_dataset/test/images',
    conf=0.25,
    save=True,
    project=PROJECT,
    name=f'{RUN_NAME}_predict'
)
print('Saved predictions to:', Path(PROJECT) / f'{RUN_NAME}_predict')


image 1/1000 /home/xd/Documents/py_codes/arrows/arrow_dataset/test/images/100_png_jpg.rf.0a108ff06b8e212bafc345c4bfa2b93f.jpg: 640x640 1 left, 1 right, 2 ups, 2.8ms
image 2/1000 /home/xd/Documents/py_codes/arrows/arrow_dataset/test/images/100_png_jpg.rf.513a923e00e3ca2fd9e3c2bbc2282609.jpg: 640x640 1 left, 1 right, 2 ups, 3.5ms
image 3/1000 /home/xd/Documents/py_codes/arrows/arrow_dataset/test/images/100_png_jpg.rf.9451fd3f48b53d82568ee41cba031342.jpg: 640x640 1 left, 1 right, 2 ups, 4.6ms
image 4/1000 /home/xd/Documents/py_codes/arrows/arrow_dataset/test/images/101_png_jpg.rf.800e7a24406593a609317c1d757dacee.jpg: 640x640 1 down, 1 left, 1 right, 1 up, 3.2ms
image 5/1000 /home/xd/Documents/py_codes/arrows/arrow_dataset/test/images/101_png_jpg.rf.dcaed33a211f607e83e223962ae0045c.jpg: 640x640 1 down, 1 left, 1 right, 1 up, 3.6ms
image 6/1000 /home/xd/Documents/py_codes/arrows/arrow_dataset/test/images/102_png_jpg.rf.51ac565dbcfd0cc5d95dd17a08061389.jpg: 640x640 1 left, 3 ups, 2.8ms
imag

## Notes
- Best weights are saved at: `runs/arrow_yolov8n/weights/best.pt`
- You can switch to a larger model by changing `MODEL_WEIGHTS` to `yolov8s.pt`/`yolov8m.pt`.
- For webcam inference later, use `best_model.predict(source=0, show=True)` in a Python script or notebook cell.